# Day 1 - PostgreSQL + pgvector
## Semantic Job Listing Search

PostgreSQL is the world's most widely deployed open-source relational database. With the `pgvector` extension, it gains native vector similarity search, meaning you can add semantic search to an existing Postgres database without introducing a new system.

**When would you reach for this?**
- You already have data in PostgreSQL
- You want vector search alongside relational filtering (by location, salary, date)
- You want to minimise operational complexity by using one database, not two

**The use case:** A job listing search engine where users can describe what they are looking for in natural language and get semantically relevant results filtered by location and salary where needed.

## 1. Setup

### Prerequisites

- PostgreSQL 16 installed locally
- `pgvector` installed from source (see book setup instructions)
- Ollama running locally with `all-minilm` model pulled

### Install Python dependencies

In [1]:
%pip install ollama==0.6.2 \
             pandas==3.0.3 \
             psycopg2-binary==2.9.12 \
             sqlalchemy==2.0.51 \
             tqdm==4.67.1 --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import ollama
import os
import random
import psycopg2
import pandas as pd

from ollama import ResponseError
from tqdm.notebook import tqdm

### Configuration

Set `NUM_JOBS` to control how many job listings to generate. The default is 50, which is enough to demonstrate semantic search. Increase to 500 or 5000 to explore performance at scale. `RANDOM_SEED` ensures the same dataset is generated each run.

In [3]:
LLM_EMBEDDING = "all-minilm"
NUM_JOBS = 500
RANDOM_SEED = 42

### Verify Ollama is running

In [4]:
ollama_ready = False

try:
    models = ollama.list()
    model_names = [m.model for m in models.models]
    assert any(LLM_EMBEDDING in m for m in model_names)
    print(f"Model '{LLM_EMBEDDING}' is ready.")
    ollama_ready = True
except ConnectionError:
    print("ERROR: Ollama is not running. Start it with: ollama serve")
except AssertionError:
    print(f"ERROR: Model not found. Run: ollama pull {LLM_EMBEDDING}")

Model 'all-minilm' is ready.


In [5]:
assert ollama_ready, "Please fix the Ollama issue above before continuing."

### Connect to PostgreSQL

In [6]:
conn = psycopg2.connect(
    dbname = "postgres",
    user   = os.environ.get("USER"),
    host   = "localhost",
    port   = 5432
)
conn.autocommit = True
cursor = conn.cursor()
print("Connected to PostgreSQL.")

Connected to PostgreSQL.


### Enable pgvector

In [7]:
cursor.execute("CREATE EXTENSION IF NOT EXISTS vector;")
print("pgvector extension enabled.")

pgvector extension enabled.


## 2. The Dataset

Rather than a hardcoded list, we generate job listings programmatically from pools of titles, companies, locations, skills and description components. This keeps the notebook concise and lets us scale to thousands of listings by changing `NUM_JOBS` above.

Each description is assembled from role-appropriate components, giving enough variation for meaningful semantic search without requiring a hand-written entry for every listing.

In [8]:
random.seed(RANDOM_SEED)

ROLES = [
    {"title": "Senior Data Engineer",      "domain": "data",       "salary_band": (130000, 160000)},
    {"title": "Machine Learning Engineer", "domain": "ml",         "salary_band": (150000, 190000)},
    {"title": "Backend Software Engineer", "domain": "backend",    "salary_band": (110000, 140000)},
    {"title": "Data Scientist",            "domain": "data",       "salary_band": (115000, 145000)},
    {"title": "DevOps Engineer",           "domain": "devops",     "salary_band": (120000, 150000)},
    {"title": "Frontend Engineer",         "domain": "frontend",   "salary_band": (105000, 135000)},
    {"title": "Data Analyst",              "domain": "data",       "salary_band": (85000,  110000)},
    {"title": "Site Reliability Engineer", "domain": "devops",     "salary_band": (130000, 165000)},
    {"title": "NLP Engineer",              "domain": "ml",         "salary_band": (140000, 175000)},
    {"title": "Platform Engineer",         "domain": "devops",     "salary_band": (125000, 155000)},
    {"title": "Analytics Engineer",        "domain": "data",       "salary_band": (110000, 140000)},
    {"title": "Computer Vision Engineer",  "domain": "ml",         "salary_band": (145000, 180000)},
    {"title": "Security Engineer",         "domain": "security",   "salary_band": (130000, 160000)},
    {"title": "Full Stack Engineer",       "domain": "backend",    "salary_band": (100000, 130000)},
    {"title": "ML Ops Engineer",           "domain": "ml",         "salary_band": (135000, 165000)},
    {"title": "Cloud Architect",           "domain": "devops",     "salary_band": (160000, 200000)},
    {"title": "Search Engineer",           "domain": "backend",    "salary_band": (130000, 160000)},
    {"title": "Data Engineer",             "domain": "data",       "salary_band": (105000, 135000)},
    {"title": "Staff Engineer",            "domain": "backend",    "salary_band": (180000, 230000)},
    {"title": "Applied Scientist",         "domain": "ml",         "salary_band": (155000, 195000)},
    {"title": "Prompt Engineer",           "domain": "ml",         "salary_band": (110000, 140000)},
    {"title": "Database Administrator",    "domain": "data",       "salary_band": (95000,  120000)},
    {"title": "Embedded Systems Engineer", "domain": "systems",    "salary_band": (115000, 145000)},
    {"title": "Rust Engineer",             "domain": "systems",    "salary_band": (140000, 175000)},
    {"title": "Kafka Engineer",            "domain": "data",       "salary_band": (130000, 160000)},
    {"title": "Vector Database Engineer",  "domain": "ml",         "salary_band": (140000, 175000)},
    {"title": "Engineering Manager",       "domain": "leadership", "salary_band": (170000, 210000)},
    {"title": "Principal Engineer",        "domain": "leadership", "salary_band": (190000, 240000)},
    {"title": "QA Engineer",               "domain": "qa",         "salary_band": (85000,  110000)},
    {"title": "Technical Writer",          "domain": "other",      "salary_band": (75000,  95000)},
]

COMPANIES = [
    "Acme Analytics", "Synapse AI", "CloudBase", "RetailIQ", "Streamline Systems",
    "Pixel Works", "FinSight", "Orbis Cloud", "LinguaTech", "CoreStack",
    "DataFlow", "VisionCore", "ShieldNet", "LaunchPad", "PulseData",
    "ModelShip", "ConnectGraph", "SkyBridge", "AppCraft", "Meridian Data",
    "AlphaEdge", "FindIt", "DataGuard", "EdgeCore", "GrowthLab",
    "SystemsIO", "InsightCo", "DeepFoundry", "DocuCraft", "ScaleUp",
    "TrustData", "BridgeIT", "StreamOps", "BuildRight", "QualityFirst",
    "NimbusData", "LLMApps", "NetLayer", "CognitiveAI", "UptimeIO",
]

LOCATIONS = [
    "New York, NY", "San Francisco, CA", "Austin, TX", "Chicago, IL",
    "Seattle, WA", "Boston, MA", "Los Angeles, CA", "Denver, CO",
    "Washington, DC", "Miami, FL", "Atlanta, GA", "Dallas, TX",
    "Portland, OR", "Minneapolis, MN", "Houston, TX", "Phoenix, AZ",
    "Remote",
]

SKILLS_BY_DOMAIN = {
    "data":      [["Python", "Spark", "SQL", "Airflow"], ["dbt", "Snowflake", "SQL", "Python"],
                  ["Kafka", "Spark", "GCP", "Python"], ["Databricks", "Delta Lake", "Python", "SQL"]],
    "ml":        [["Python", "PyTorch", "Kubernetes", "MLflow"], ["Python", "Transformers", "spaCy", "BERT"],
                  ["Python", "OpenCV", "PyTorch", "CUDA"], ["Python", "Weaviate", "Pinecone", "Embeddings"]],
    "backend":   [["Go", "PostgreSQL", "Docker", "REST APIs"], ["React", "Node.js", "PostgreSQL", "Docker"],
                  ["Java", "Spring Boot", "PostgreSQL", "Microservices"], ["Elasticsearch", "Python", "Lucene", "Vector Search"]],
    "frontend":  [["React", "TypeScript", "CSS", "GraphQL"], ["React", "TypeScript", "Tailwind", "REST APIs"]],
    "devops":    [["Terraform", "AWS", "Kubernetes", "CI/CD"], ["Linux", "Prometheus", "Grafana", "Python"],
                  ["Kubernetes", "Helm", "Python", "AWS"], ["AWS", "Azure", "Terraform", "Solution Design"]],
    "security":  [["Python", "SIEM", "AWS Security", "Penetration Testing"]],
    "systems":   [["C", "C++", "RTOS", "ARM"], ["Rust", "WebAssembly", "Networking", "Linux"]],
    "leadership":[["System Design", "Python", "Java", "Leadership"], ["Leadership", "Agile", "Python", "System Design"]],
    "qa":        [["Selenium", "Python", "Test Automation", "CI/CD"]],
    "other":     [["Technical Writing", "Markdown", "API Documentation", "Git"]],
}

DESCRIPTION_TEMPLATES = {
    "data": [
        "Build and maintain {system} for a {company_type} specializing in {domain}. Work closely with {team} to ensure data quality and reliability.",
        "Design scalable {system} on {platform}. Enable {team} to access clean, timely data for analytics and decision-making.",
        "Own the {system} that powers company-wide reporting. Collaborate with {team} to model and transform raw data into reliable datasets.",
    ],
    "ml": [
        "Develop and deploy {system} at scale for a {company_type}. Collaborate with {team} to translate experiments into production systems.",
        "Build {system} for real-time inference serving millions of users. Work with {team} on model evaluation, deployment and monitoring.",
        "Apply state-of-the-art {system} to real-world product problems. Prototype new approaches and collaborate with {team} to ship AI features.",
    ],
    "backend": [
        "Develop robust {system} for a {company_type}. Own features end to end from design through deployment and monitoring.",
        "Build and maintain {system} serving high traffic workloads. Work with {team} to improve reliability, performance and developer experience.",
        "Design and ship {system} for a fast-growing {company_type}. Comfortable owning features across the full stack.",
    ],
    "frontend": [
        "Build performant {system} for a {company_type}. Work closely with {team} to deliver polished, accessible user experiences.",
        "Develop and maintain {system} used by millions of users. Collaborate with {team} on design, performance and accessibility.",
    ],
    "devops": [
        "Manage {system} for a {company_type}. Drive automation, reliability and cost optimisation across engineering teams.",
        "Design and operate {system} at scale. Improve developer experience and lead incident response for critical infrastructure.",
        "Build {system} that enables engineering teams to ship faster and safer. Own reliability, observability and deployment pipelines.",
    ],
    "security": [
        "Protect {system} from security threats at a {company_type}. Conduct vulnerability assessments, respond to incidents and implement controls.",
    ],
    "systems": [
        "Develop {system} optimised for performance and low-level efficiency. Work close to the hardware on a {company_type} product.",
        "Build high-performance {system} for a developer tools {company_type}. Focus on correctness, safety and throughput.",
    ],
    "leadership": [
        "Provide technical leadership across engineering teams at a {company_type}. Define architecture standards and mentor senior engineers.",
        "Lead a team of engineers building {system} for a {company_type}. Own technical delivery, team growth and cross-functional collaboration.",
    ],
    "qa": [
        "Build and maintain automated {system} for a {company_type}. Work with developers to shift quality left and catch regressions early.",
    ],
    "other": [
        "Write clear, accurate {system} for a {company_type}. Work closely with engineers to document APIs, SDKs and integration guides.",
    ],
}

SYSTEMS_BY_DOMAIN = {
    "data":       ["data pipelines", "ETL workflows", "streaming data infrastructure", "data warehouse models", "lakehouse pipelines"],
    "ml":         ["machine learning models", "NLP pipelines", "computer vision systems", "recommendation engines", "vector search infrastructure"],
    "backend":    ["backend services", "REST APIs", "microservices", "search infrastructure", "event-driven systems"],
    "frontend":   ["user interfaces", "web applications", "component libraries", "design systems"],
    "devops":     ["cloud infrastructure", "Kubernetes clusters", "CI/CD pipelines", "developer platforms", "observability systems"],
    "security":   ["cloud and application security controls", "security monitoring systems"],
    "systems":    ["firmware and embedded software", "systems software", "networking infrastructure"],
    "leadership": ["engineering teams", "platform infrastructure", "core systems"],
    "qa":         ["test automation frameworks", "quality assurance pipelines"],
    "other":      ["technical documentation", "developer guides and API references"],
}

COMPANY_TYPES = ["fintech", "healthtech", "SaaS", "e-commerce", "media", "enterprise software", "startup", "data platform"]
PLATFORMS     = ["AWS", "GCP", "Azure", "Databricks", "Snowflake"]
TEAMS         = ["data scientists", "analysts", "product managers", "ML engineers", "backend engineers", "platform teams"]
DOMAINS       = ["real-time analytics", "fraud detection", "personalisation", "demand forecasting", "search", "observability"]

def generate_description(domain: str) -> str:
    template = random.choice(DESCRIPTION_TEMPLATES[domain])
    return template.format(
        system       = random.choice(SYSTEMS_BY_DOMAIN[domain]),
        company_type = random.choice(COMPANY_TYPES),
        platform     = random.choice(PLATFORMS),
        team         = random.choice(TEAMS),
        domain       = random.choice(DOMAINS),
    )

def generate_job_listings(n: int) -> list:
    listings = []
    for _ in range(n):
        role = random.choice(ROLES)
        domain = role["domain"]
        sal_min, sal_max = role["salary_band"]
        # Add a small random offset so listings of the same role vary
        offset = random.choice([-10000, -5000, 0, 5000, 10000])
        listings.append({
            "title":       role["title"],
            "company":     random.choice(COMPANIES),
            "location":    random.choice(LOCATIONS),
            "salary_min":  sal_min + offset,
            "salary_max":  sal_max + offset,
            "skills":      random.choice(SKILLS_BY_DOMAIN[domain]),
            "description": generate_description(domain),
        })
    return listings

job_listings = generate_job_listings(NUM_JOBS)
df = pd.DataFrame(job_listings)
print(f"Generated {len(df)} job listings")
df.head()

Generated 500 job listings


,title,company,location,salary_min,salary_max,skills,description
0,Prompt Engineer,Synapse AI,"Washington, DC",100000,130000,"[Python, Transformers, spaCy, BERT]",Develop and deploy NLP pipelines at scale for ...
1,Full Stack Engineer,Synapse AI,"Austin, TX",90000,120000,"[React, Node.js, PostgreSQL, Docker]",Develop robust event-driven systems for a fint...
2,Prompt Engineer,InsightCo,"Denver, CO",120000,150000,"[Python, Weaviate, Pinecone, Embeddings]",Apply state-of-the-art computer vision systems...
3,Analytics Engineer,CoreStack,"Los Angeles, CA",110000,140000,"[Kafka, Spark, GCP, Python]",Build and maintain data pipelines for a startu...
4,Applied Scientist,CloudBase,"Houston, TX",155000,195000,"[Python, PyTorch, Kubernetes, MLflow]",Build machine learning models for real-time in...


## 3. Create the Table

In [9]:
cursor.execute("DROP TABLE IF EXISTS job_listings;")

cursor.execute("""
    CREATE TABLE job_listings (
        id          SERIAL PRIMARY KEY,
        title       TEXT,
        company     TEXT,
        location    TEXT,
        salary_min  INTEGER,
        salary_max  INTEGER,
        skills      TEXT[],
        description TEXT,
        embedding   vector(384)
    );
""")

print("Table created.")

Table created.


## 4. Generate Embeddings and Load Data

We embed the job description for each listing using `all-minilm` via Ollama. The `all-minilm` model produces 384-dimensional vectors.

In [10]:
def get_embedding(text: str) -> list:
    response = ollama.embeddings(model = LLM_EMBEDDING, prompt = text)
    return response["embedding"]

# Test
test_embedding = get_embedding("senior data engineer with Python and Spark")
print(f"Embedding dimensions: {len(test_embedding)}")

Embedding dimensions: 384


In [11]:
for job in tqdm(job_listings, desc = "Inserting listings"):
    embedding = get_embedding(job["description"])
    cursor.execute("""
        INSERT INTO job_listings (title, company, location, salary_min, salary_max, skills, description, embedding)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    """, (
        job["title"],
        job["company"],
        job["location"],
        job["salary_min"],
        job["salary_max"],
        job["skills"],
        job["description"],
        str(embedding)
    ))

print(f"\nInserted {len(job_listings)} job listings with embeddings.")

Inserting listings:   0%|          | 0/500 [00:00<?, ?it/s]


Inserted 500 job listings with embeddings.


## 5. Create a Vector Index

pgvector supports two index types: IVFFlat and HNSW. HNSW gives better recall and is the recommended default. For large datasets, build the index after loading data rather than before - this is significantly faster.

In [12]:
cursor.execute("""
    CREATE INDEX ON job_listings
    USING hnsw (embedding vector_cosine_ops);
""")

print("HNSW index created.")

HNSW index created.


## 6. Semantic Search

We embed the user's query and find the most similar job descriptions using cosine similarity.

In [13]:
def search_jobs(query: str, top_k: int = 5):
    query_embedding = get_embedding(query)
    cursor.execute("""
        SELECT title, company, location, salary_min, salary_max,
               1 - (embedding <=> %s::vector) AS similarity
        FROM job_listings
        ORDER BY embedding <=> %s::vector
        LIMIT %s;
    """, (str(query_embedding), str(query_embedding), top_k))

    results = cursor.fetchall()
    print(f"\nQuery: '{query}'\n")
    for row in results:
        title, company, location, sal_min, sal_max, similarity = row
        print(f"  {title} @ {company} - {location}")
        print(f"  ${sal_min:,} - ${sal_max:,} | Similarity: {similarity:.3f}")
        print()

In [14]:
search_jobs("I want to work on machine learning models in production")


Query: 'I want to work on machine learning models in production'

  ML Ops Engineer @ RetailIQ - Phoenix, AZ
  $135,000 - $165,000 | Similarity: 0.717

  Machine Learning Engineer @ EdgeCore - Portland, OR
  $155,000 - $195,000 | Similarity: 0.700

  Prompt Engineer @ InsightCo - Portland, OR
  $110,000 - $140,000 | Similarity: 0.692

  Computer Vision Engineer @ GrowthLab - Austin, TX
  $140,000 - $175,000 | Similarity: 0.669

  ML Ops Engineer @ AlphaEdge - Austin, TX
  $140,000 - $170,000 | Similarity: 0.635



In [15]:
search_jobs("looking for a data pipeline and ETL role")


Query: 'looking for a data pipeline and ETL role'

  Senior Data Engineer @ SystemsIO - Remote
  $120,000 - $150,000 | Similarity: 0.679

  Data Engineer @ NetLayer - Minneapolis, MN
  $115,000 - $145,000 | Similarity: 0.679

  Kafka Engineer @ TrustData - San Francisco, CA
  $140,000 - $170,000 | Similarity: 0.675

  Kafka Engineer @ GrowthLab - Remote
  $130,000 - $160,000 | Similarity: 0.671

  Kafka Engineer @ Meridian Data - Chicago, IL
  $135,000 - $165,000 | Similarity: 0.667



In [16]:
search_jobs("I enjoy working close to the hardware and embedded systems")


Query: 'I enjoy working close to the hardware and embedded systems'

  Embedded Systems Engineer @ DataFlow - Portland, OR
  $105,000 - $135,000 | Similarity: 0.502

  Rust Engineer @ RetailIQ - Washington, DC
  $135,000 - $170,000 | Similarity: 0.478

  Embedded Systems Engineer @ LinguaTech - Chicago, IL
  $115,000 - $145,000 | Similarity: 0.478

  Rust Engineer @ FindIt - San Francisco, CA
  $150,000 - $185,000 | Similarity: 0.468

  Embedded Systems Engineer @ CloudBase - Austin, TX
  $115,000 - $145,000 | Similarity: 0.468



## 7. The Relational Advantage - Filtered Search

This is where PostgreSQL genuinely shines over a dedicated vector database. We can combine semantic similarity with standard SQL filters - location, salary range, required skills - in a single query. No separate metadata filtering step, no second system.

In [17]:
def search_jobs_filtered(query: str, location: str = None, min_salary: int = None, top_k: int = 5):
    query_embedding = get_embedding(query)
    embedding_str   = str(query_embedding)

    filters = []
    filter_params = []

    if location:
        filters.append("location ILIKE %s")
        filter_params.append(f"%{location}%")
    if min_salary:
        filters.append("salary_min >= %s")
        filter_params.append(min_salary)

    where_clause = "WHERE " + " AND ".join(filters) if filters else ""

    # First embedding param goes before the WHERE filters; second after (ORDER BY)
    params = [embedding_str] + filter_params + [embedding_str, top_k]

    cursor.execute(f"""
        SELECT title, company, location, salary_min, salary_max,
               1 - (embedding <=> %s::vector) AS similarity
        FROM job_listings
        {where_clause}
        ORDER BY embedding <=> %s::vector
        LIMIT %s;
    """, params)

    results = cursor.fetchall()
    label = f"query = '{query}'"
    if location: label += f", location = '{location}'"
    if min_salary: label += f", min_salary = ${min_salary:,}"
    print(f"\n{label}\n")
    for row in results:
        title, company, loc, sal_min, sal_max, similarity = row
        print(f"  {title} @ {company} - {loc}")
        print(f"  ${sal_min:,} - ${sal_max:,} | Similarity: {similarity:.3f}")
        print()

In [18]:
# Semantic search filtered to New York with a minimum salary
search_jobs_filtered(
    "technical leadership and system design",
    location = "New York",
    min_salary = 130000
)


query = 'technical leadership and system design', location = 'New York', min_salary = $130,000

  Engineering Manager @ GrowthLab - New York, NY
  $175,000 - $215,000 | Similarity: 0.449

  Engineering Manager @ BridgeIT - New York, NY
  $175,000 - $215,000 | Similarity: 0.449

  Site Reliability Engineer @ DocuCraft - New York, NY
  $140,000 - $175,000 | Similarity: 0.336

  Search Engineer @ Orbis Cloud - New York, NY
  $135,000 - $165,000 | Similarity: 0.294

  NLP Engineer @ Orbis Cloud - New York, NY
  $135,000 - $170,000 | Similarity: 0.272



In [19]:
# Remote roles in AI and machine learning
search_jobs_filtered(
    "artificial intelligence and language models",
    location = "Remote"
)


query = 'artificial intelligence and language models', location = 'Remote'

  Applied Scientist @ InsightCo - Remote
  $145,000 - $185,000 | Similarity: 0.343

  Prompt Engineer @ DocuCraft - Remote
  $110,000 - $140,000 | Similarity: 0.313

  Vector Database Engineer @ Pixel Works - Remote
  $140,000 - $175,000 | Similarity: 0.278

  Applied Scientist @ PulseData - Remote
  $165,000 - $205,000 | Similarity: 0.267

  Computer Vision Engineer @ LLMApps - Remote
  $135,000 - $170,000 | Similarity: 0.237



## 8. What You'd Hit in Production

- **Index build time** - HNSW indexes are built at insert time, which slows bulk loads. For large datasets, build the index after loading data.
- **Dimensionality limit** - pgvector supports up to 2,000 dimensions. Most embedding models are well within this, but larger models (e.g. text-embedding-3-large at 3,072 dims) require dimensionality reduction.
- **Approximate vs exact search** - HNSW is approximate. For exact nearest neighbour search, omit the index and use a sequential scan, but this does not scale.
- **Connection pooling** - use PgBouncer or a pooling layer in production; vector queries can be memory-intensive.
- **Hosting** - Supabase and Neon both offer hosted Postgres with pgvector if you prefer not to manage the server yourself.

## 9. When to Look Elsewhere

PostgreSQL + pgvector is a strong default choice, but consider a dedicated vector database if:

- You are storing tens of millions of vectors and need sub-10ms retrieval at scale
- You need advanced filtering on high-cardinality metadata without index performance trade-offs
- Your team has no existing Postgres footprint and does not want to manage it
- You need multi-tenancy, namespacing or built-in embedding model integrations out of the box

For most applications - especially those that already run on Postgres - `pgvector` is the right place to start.

## Cleanup

In [20]:
cursor.close()
conn.close()
print("Connection closed.")

Connection closed.
